# 68 · Organize direct Oλ sensitivity results
Run after both notebook 67 analyses finish. This organizer reads their saved tables, summarizes pairs of patient estimates, and exports CSVs. Replicate intervals are empirical resampling intervals. Survival intervals are posterior credible intervals. Reconstruction intervals are within-dataset patient-bootstrap intervals around the held-out performance estimates.

In [ ]:
from google.colab import drive
from pathlib import Path
from itertools import combinations, product
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
SURV = DATA/'bayesian_Olambda_sensitivity_survival/direct_reciprocal_adjusted_midpoint_v1'
RECON = DATA/'direct_Olambda_reconstruction_from_TCAT_and_atlas'
DIRECT = DATA/'olambda_reciprocal_direct_no_asymptote_floor_v1/patient_olambda_asymptotes.csv'
OUT = DATA/'organized_direct_Olambda_sensitivity_results/tables'
RECON_RUN = None  # Set to a specific completed run directory if there is more than one.
experiments = {'patient-bootstrap':'patient_bayesian_bootstrap',
               'pooled-synthetic':'atlas_hard_label_pooled_synthetic'}
transition = 'smooth_transition_Q10_Q90_midpoint'
keys = ['Dataset','sample_id']
tables = {}
sources = []
def read(path, **kwargs):
    sources.append(path)
    return pd.read_csv(path, **kwargs)
def manifest(path):
    sources.append(path)
    return json.loads(path.read_text())

index = read(SURV/'sensitivity_survival_model_comparison.csv').set_index(['experiment','fit'],verify_integrity=True)
needed = set(product(experiments,['continuous',transition]))
if needed != set(index.index):
    raise ValueError(f'Expected four completed survival fits. Missing: {needed-set(index.index)}')
runs = [Path(RECON_RUN)] if RECON_RUN else sorted(p.parent for p in RECON.glob('*/reconstruction_summary.csv'))
if len(runs)!=1:
    raise ValueError(f'Found {len(runs)} completed reconstruction runs in {RECON}. Finish notebook 67 or set RECON_RUN. Candidates: {runs}')
run = runs[0]
assert manifest(run/'config.json')['target_scale']=='direct'
print('Reconstruction:',run,'\nSurvival:',SURV)

In [ ]:
direct = read(DIRECT).set_index(keys,verify_integrity=True)
assert direct.olambda_scale.eq('reciprocal').all() and direct.model.eq('bounded_direct_no_asymptote_floor').all()
assert len(direct)==163 and direct.index.get_level_values('Dataset').nunique()==8
wide = direct[['O_lambda_inf_central']].rename(columns={'O_lambda_inf_central':'direct_asymptotic_Olambda'})
rep_tables, summaries = [], []
for experiment,folder in experiments.items():
    prepared = SURV/folder/'prepared_inputs'
    rep = read(prepared/'reciprocal_replicates.csv')
    assert rep.experiment.eq(experiment).all() and set(rep.replicate)==set(range(25))
    assert not rep.duplicated(keys+['replicate']).any()
    np.testing.assert_allclose(rep.O_lambda,1/rep.T_lambda)
    assert rep.O_lambda.between(0,1,inclusive='neither').all()
    group = rep.groupby(keys)
    summary = group.agg(replicates=('replicate','size'),native_cells=('n_cells_native','first'),
                        O_lambda_mean=('O_lambda','mean'),O_lambda_median=('O_lambda','median'),
                        O_lambda_sd=('O_lambda','std'))
    assert summary.replicates.eq(25).all() and summary.index.difference(direct.index).empty
    summary['O_lambda_q025'] = group.O_lambda.quantile(.025)
    summary['O_lambda_q975'] = group.O_lambda.quantile(.975)
    est = read(prepared/'patient_estimates.csv').set_index(keys,verify_integrity=True)
    assert summary.index.equals(est.sort_index().index)
    np.testing.assert_allclose(summary.O_lambda_mean,est.loc[summary.index,'O_lambda_inf_central'])
    wide[experiment] = summary.O_lambda_mean
    summaries.append(summary.reset_index().assign(experiment=experiment))
    rep_tables.append(rep)
tables['all_sensitivity_replicates'] = pd.concat(rep_tables,ignore_index=True)
estimates = tables['patient_sensitivity_estimates'] = pd.concat(summaries,ignore_index=True)
tables['patient_estimate_agreement'] = wide.reset_index()
pairs = []
for first,second in combinations(wide.columns,2):
    shared = wide[[first,second]].dropna()
    pairs.append(dict(first_estimate=first,second_estimate=second,patients=len(shared),
                      Spearman_r=shared[first].corr(shared[second],method='spearman')))
tables['estimate_agreement_summary'] = pd.DataFrame(pairs)
tables['sensitivity_estimate_audit'] = estimates.groupby('experiment',as_index=False).agg(
    patients=('sample_id','size'), datasets=('Dataset','nunique'),replicates=('replicates','sum'),
    median_within_patient_sd=('O_lambda_sd','median'))
display(tables['sensitivity_estimate_audit'], tables['estimate_agreement_summary'])

In [ ]:
collected = {}
for (experiment,fit),entry in index.iterrows():
    dest = SURV/experiments[experiment]/fit/Path(entry.output_dir).name
    m = manifest(dest/'run_manifest.json')
    assert m['adjustments']=={'Treatment':'axicel','Disease':'lymphoma'} and m['n_particles']==25
    if fit==transition:
        assert m['hazard_activation_offset']==0.5
    result = read(dest/'model_result.csv')
    cohort = read(dest/'analysis_cohort.csv',usecols=keys+['event_observed'])
    ids = wide[experiment].dropna().index
    assert cohort.set_index(keys,verify_integrity=True).sort_index().index.equals(ids.sort_values())
    assert len(result)==1 and result.patients.iloc[0]==len(ids)
    expected_mean = wide.loc[ids,experiment]
    saved = read(dest/'analysis_cohort.csv',usecols=keys+['O_lambda_inf_central']).set_index(keys)
    np.testing.assert_allclose(saved.loc[ids,'O_lambda_inf_central'],expected_mean)
    pieces = {'sensitivity_survival_results':result,
              'survival_dataset_effects':read(dest/'dataset_effects.csv'),
              'survival_adjustment_effects':read(dest/'adjustment_effects.csv'),
              'survival_cohort_counts':cohort.groupby('Dataset',as_index=False).agg(
                  patients=('sample_id','size'),events=('event_observed','sum'))}
    if fit==transition:
        pieces['transition_patient_memberships'] = read(dest/'patient_regime_probabilities.csv')
    for name,frame in pieces.items():
        collected.setdefault(name,[]).append(frame.assign(experiment=experiment,fit=fit,run_id=dest.name))
for name,parts in collected.items():
    tables[name] = pd.concat(parts,ignore_index=True)
display(tables['sensitivity_survival_results'][['experiment','fit','patients','events','HR_median',
    'HR_q025','HR_q975','P_all_datasets_same_direction','crossover_median','divergences','max_rhat']])

In [ ]:
target_table = read(run/'reconstruction_targets.csv').set_index(keys,verify_integrity=True).sort_index()
assert target_table.index.equals(wide.sort_index().index)
targets = ['direct_asymptotic_Olambda','patient_bootstrap_Olambda']
np.testing.assert_allclose(target_table[targets],wide.loc[target_table.index,['direct_asymptotic_Olambda','patient-bootstrap']])
summary = read(run/'reconstruction_summary.csv')
representations = ['TCAT GEP','Atlas cell type','TCAT GEP + Atlas']
models = ['Dataset mean','Ridge','Elastic net','PLS','Gradient boosting']
group_keys = ['target','representation','model']
assert set(summary.set_index(group_keys,verify_integrity=True).index)==set(product(targets,representations,models))
assert summary.patients.eq(len(target_table)).all()
tables['reconstruction_R2_and_Spearman'] = summary
tables['best_reconstruction_models'] = (summary.loc[summary.model.ne('Dataset mean')]
    .sort_values(['R2','model'],ascending=[False,True]).drop_duplicates(['target','representation']))
tables['reconstruction_targets'] = target_table.reset_index()
for name in ['patient_fold_assignments','patient_cross_validated_predictions',
             'repeated_cross_validated_predictions','nested_tuning_results']:
    tables[name] = read(run/(name+'.csv'))
display(tables['best_reconstruction_models'][['target','representation','model','R2','R2_q025',
        'R2_q975','dataset_adjusted_Spearman_r','r_q025','r_q975']])

In [ ]:
# Publish the CSV tables only after all input checks pass.
OUT.mkdir(parents=True,exist_ok=True)
tables['source_inventory'] = pd.DataFrame([
    dict(source=str(p.relative_to(DATA)) if p.is_relative_to(DATA) else str(p),
         sha256=hashlib.sha256(p.read_bytes()).hexdigest()) for p in dict.fromkeys(sources)])
inventory = []
for name,table in tables.items():
    path = OUT/(name+'.csv')
    table.to_csv(path,index=False)
    inventory.append(dict(table=path.name,rows=len(table),columns=len(table.columns)))
inventory = pd.DataFrame(inventory)
inventory.to_csv(OUT/'table_inventory.csv',index=False)
display(inventory)
print('Saved tables:',OUT)